# humanoid-video-to-3d — phone video → 3D point cloud (Colab)

GPU path using **MASt3R** (the more accurate successor to DUSt3R). No COLMAP, no provided camera calibration.

**Before you run:**
1. `Runtime → Change runtime type → Hardware accelerator: **T4 GPU**`
2. `Runtime → Run all`, then upload a short clip when prompted.

Pipeline: **upload video → extract frames → blur filter → MASt3R → semantic labels → view → save**

In [ ]:
!nvidia-smi -L

In [ ]:
REPO_URL = "https://github.com/maheswariridhi/humanoid-video-to-3d.git"  # <-- change if your repo differs

import os, sys
if not os.path.isdir("humanoid-video-to-3d"):
    !git clone $REPO_URL
%cd humanoid-video-to-3d
!pip install -q -r requirements.txt

# MASt3R — the reconstruction engine (clones DUSt3R as a submodule dependency).
if not os.path.isdir("mast3r"):
    !git clone --recursive https://github.com/naver/mast3r
    !pip install -q -r mast3r/requirements.txt
    !pip install -q -r mast3r/dust3r/requirements.txt
sys.path += ["mast3r", "mast3r/dust3r"]

## 2. Upload your video
A 10–30 s slow walk-around of a small room or a desk works best.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print("uploaded:", video_path)

## 3. Prep frames (CPU): extract → deblur → thin to fit GPU memory

In [ ]:
from v3d import frames

frames.extract(video_path, "out/images", fps=2)
frames.filter_blurry("out/images")
frames.subsample("out/images", max_frames=25)  # denser coverage; stays on MASt3R's "complete" graph (<=25)

## 4. Reconstruct (GPU): MASt3R → coloured point cloud
First run downloads the model (~2 GB).

In [ ]:
from v3d import reconstruct

# Raise min_conf (e.g. 3, 5) for fewer but cleaner points.
rec = reconstruct.run("out/images", device="cuda")

## 5. Semantic labels in 3D
Segment each frame (SegFormer / ADE20K), lift the labels onto the 3D points (so
they're aligned with the geometry), then voxel-majority-vote for multi-view
consistency. Furniture classes are coloured; everything else stays grey.

In [ ]:
!pip install -q transformers
from v3d import semantics, pointcloud

labeled = semantics.label(rec, device="cuda")     # default = common furniture classes
print(semantics.summary(labeled))

sem_colors = semantics.semantic_colors(labeled)
pointcloud.show(labeled.points, sem_colors)                          # coloured by class
pointcloud.save_ply(labeled.points, sem_colors, "out/point_cloud_semantic.ply")

## 6. View inline + save outputs
Renders the RGB cloud here and writes `point_cloud.ply`, a standalone
`preview.html`, and `report.json` into `out/` (download from the Files panel when needed).

In [ ]:
import json
from pathlib import Path
from v3d import pointcloud

# Show the cloud inline + save a standalone HTML you can reopen in any browser.
fig = pointcloud.show(rec.points, rec.colors)
fig.write_html("out/preview.html")
pointcloud.save_ply(rec.points, rec.colors, "out/point_cloud.ply")

report = {
    "num_frames_used": len(list(Path("out/images").glob("frame_*.jpg"))),
    "num_points": int(len(rec.points)),
    "outputs": sorted(str(p) for p in Path("out").glob("*") if p.is_file()),
}
Path("out/report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("\nSaved under out/. Grab any file from the Files panel (folder icon on the left) -",
      "e.g. right-click point_cloud.ply -> Download.")